# 19 - Agent Memory

Memory is not simply 'put the entire chat into the prompt forever'.

This notebook separates:
- short-term/thread memory
- long-term/cross-thread memory
- working memory
- user preferences
- deletion and retention policy

Current LangGraph documentation describes **checkpointers** for thread-scoped state and **stores** for long-term data across threads. citeturn114600search0

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langgraph

## 1. Short-term memory

Short-term memory is current thread state:
```text
conversation + working state
```

It lets an agent continue a conversation or resume after a pause.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END

def remember_node(state: MessagesState):
    return {'messages': [
        {'role': 'ai', 'content': 'I can continue using this thread state.'}
    ]}

builder = StateGraph(MessagesState)
builder.add_node('remember', remember_node)
builder.add_edge(START, 'remember')
builder.add_edge('remember', END)

graph = builder.compile(checkpointer=InMemorySaver())

config = {'configurable': {'thread_id': 'user-1'}}
result = graph.invoke(
    {'messages': [{'role': 'user', 'content': 'Remember this thread.'}]},
    config,
)

print(result['messages'][-1].content)

## 2. Long-term memory

Long-term memory is application data that should survive across conversations.

Examples:
- preferred language
- stable user preference
- previously approved setting
- durable account fact

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(
    ('users', 'u-123'),
    'preferences',
    {'language': 'Python', 'difficulty': 'beginner'},
)

item = store.get(('users', 'u-123'), 'preferences')
print(item.value)

## 3. Memory policy

Not everything should be remembered.

Define:
- what can be stored
- why
- retention
- deletion
- who can access it
- whether the user can correct it

In [ ]:
memory_policy = {
    'store': ['stable preferences'],
    'avoid': ['temporary secrets', 'unnecessary sensitive data'],
    'support_delete': True,
    'namespace_by_user': True,
}
print(memory_policy)

## 4. Memory retrieval

Long-term memory becomes useful only when the right memory is retrieved at the right time.

Pattern:
```text
new task
 ↓
retrieve relevant memory
 ↓
agent uses it
 ↓
optionally update memory
```

## 5. Memory mistakes

Common problems:
- stale memory
- wrong memory
- memory leakage between users
- over-storage
- prompt bloat

So memory itself needs evaluation.

## 6. Evaluation and practice

Measure:
- memory recall accuracy
- wrong-memory rate
- thread isolation
- deletion correctness
- usefulness

Practice:
1. store user preferences
2. retrieve them in a second thread
3. delete them
4. verify they are gone.

## Evaluation

Evaluation should include privacy/security checks in addition to recall quality.

Improve with:
- user namespaces
- deletion APIs
- retention policies
- durable stores
- explicit memory write rules.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/